# 🔬 Lab 1: Data Preprocessing
## Week 5: Machine Learning with Scikit-learn

---

### 📚 Learning Objectives
By the end of this lab, you will be able to:
- Handle missing data using various imputation techniques
- Apply One-Hot Encoding and Label Encoding for categorical variables
- Perform feature scaling using Standardization and Normalization
- Detect and handle outliers using IQR and Z-score methods
- Build a complete preprocessing pipeline using scikit-learn

### ⏱️ Estimated Time: 60-75 minutes

---

**Remember:** Data preprocessing is the most critical step in ML. 70-80% of a data scientist's time is spent here!

> 🎯 **Key Principle:** Garbage in = Garbage out!

## Part 1: Setup and Data Loading

First, let's import all necessary libraries and create our sample dataset.

In [ ]:
# ============================================================
# STEP 1: Import Required Libraries
# ============================================================

# Core data manipulation libraries
import numpy as np                    # For numerical operations
import pandas as pd                   # For data manipulation

# Visualization libraries
import matplotlib.pyplot as plt       # For creating plots
import seaborn as sns                 # For statistical visualizations

# Scikit-learn preprocessing tools
from sklearn.preprocessing import (
    StandardScaler,                   # For Z-score standardization
    MinMaxScaler,                     # For Min-Max normalization
    LabelEncoder,                     # For encoding categorical labels
    OneHotEncoder                     # For one-hot encoding
)

# Imputation for handling missing values
from sklearn.impute import SimpleImputer

# For building preprocessing pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# For train-test splitting
from sklearn.model_selection import train_test_split

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)

print("✅ All libraries imported successfully!")

In [ ]:
# ============================================================
# STEP 2: Create Sample Dataset
# ============================================================
# We'll create a ride-sharing dataset similar to Uber/Careem
# This dataset intentionally contains issues that need preprocessing

np.random.seed(42)  # For reproducibility

# Create dataset with 500 ride records
n_samples = 500

data = {
    # Numerical features
    'distance_km': np.random.uniform(1, 50, n_samples),
    'duration_minutes': np.random.uniform(5, 120, n_samples),
    'fare_pkr': np.random.uniform(100, 5000, n_samples),
    'driver_rating': np.random.uniform(3.0, 5.0, n_samples),
    'surge_multiplier': np.random.choice([1.0, 1.2, 1.5, 1.8, 2.0], n_samples),
    
    # Categorical features
    'city': np.random.choice(['Karachi', 'Lahore', 'Islamabad', 'Rawalpindi', 'Faisalabad'], n_samples),
    'vehicle_type': np.random.choice(['Bike', 'Mini', 'Go', 'Premium'], n_samples),
    'payment_method': np.random.choice(['Cash', 'Card', 'Wallet'], n_samples),
    'time_of_day': np.random.choice(['Morning', 'Afternoon', 'Evening', 'Night'], n_samples),
    
    # Target variable (ride completed successfully?)
    'ride_completed': np.random.choice([0, 1], n_samples, p=[0.15, 0.85])
}

df = pd.DataFrame(data)

# Introduce MISSING VALUES (realistic scenario)
# About 5-10% missing in some columns
missing_indices = np.random.choice(n_samples, size=50, replace=False)
df.loc[missing_indices[:25], 'driver_rating'] = np.nan
df.loc[missing_indices[25:], 'duration_minutes'] = np.nan

# Introduce OUTLIERS (realistic scenario)
# Some extreme values in distance and fare
outlier_indices = np.random.choice(n_samples, size=10, replace=False)
df.loc[outlier_indices[:5], 'distance_km'] = np.random.uniform(200, 500, 5)  # Unrealistic distances
df.loc[outlier_indices[5:], 'fare_pkr'] = np.random.uniform(15000, 25000, 5)  # Extremely high fares

print("✅ Dataset created successfully!")
print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")

## Part 2: Exploratory Data Analysis (EDA)

Before preprocessing, we must understand our data!

In [ ]:
# ============================================================
# STEP 3: Explore the Dataset
# ============================================================

print("📋 First 5 rows of the dataset:")
print("=" * 80)
df.head()

In [ ]:
# Check data types and non-null counts
print("\n📊 Dataset Info:")
print("=" * 80)
df.info()

In [ ]:
# Statistical summary of numerical columns
print("\n📈 Statistical Summary:")
print("=" * 80)
df.describe()

In [ ]:
# ============================================================
# STEP 4: Check for Missing Values
# ============================================================

print("\n🔍 Missing Values Analysis:")
print("=" * 80)

# Count missing values per column
missing_counts = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_percent.round(2)
})

# Show only columns with missing values
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# ============================================================
# STEP 5: Visualize Missing Data Pattern
# ============================================================

# Create a heatmap of missing values
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd')
plt.title('Missing Values Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Columns')
plt.tight_layout()
plt.show()

print("💡 Yellow/Red indicates missing values")

## Part 3: Handling Missing Values

Three main strategies:
1. **Drop** - Remove rows/columns with missing values
2. **Impute** - Fill with mean, median, or mode
3. **Predict** - Use ML to predict missing values (advanced)

In [ ]:
# ============================================================
# STEP 6: Impute Missing Values
# ============================================================

# Create a copy to preserve original data
df_cleaned = df.copy()

# METHOD 1: Simple Imputation with Mean (for numerical columns)
# Mean imputation is suitable when data is normally distributed

print("Before Imputation:")
print(f"  driver_rating missing: {df_cleaned['driver_rating'].isnull().sum()}")
print(f"  duration_minutes missing: {df_cleaned['duration_minutes'].isnull().sum()}")

# Using SimpleImputer from scikit-learn
# strategy options: 'mean', 'median', 'most_frequent', 'constant'

numerical_imputer = SimpleImputer(strategy='mean')

# Fit and transform the columns with missing values
df_cleaned['driver_rating'] = numerical_imputer.fit_transform(
    df_cleaned[['driver_rating']]
)

# For duration, let's use median (more robust to outliers)
median_imputer = SimpleImputer(strategy='median')
df_cleaned['duration_minutes'] = median_imputer.fit_transform(
    df_cleaned[['duration_minutes']]
)

print("\nAfter Imputation:")
print(f"  driver_rating missing: {df_cleaned['driver_rating'].isnull().sum()}")
print(f"  duration_minutes missing: {df_cleaned['duration_minutes'].isnull().sum()}")

print("\n✅ Missing values handled successfully!")

In [ ]:
# ============================================================
# 💡 YOUR TURN: Practice Exercise 1
# ============================================================

# TODO: Create a new imputer using 'most_frequent' strategy
# This is useful for categorical columns

# Uncomment and complete the code below:

# mode_imputer = SimpleImputer(strategy='_______')
# test_data = pd.DataFrame({'category': ['A', 'B', np.nan, 'A', 'A', np.nan]})
# imputed_data = mode_imputer.fit_transform(test_data)
# print("Imputed values:", imputed_data.flatten())

## Part 4: Encoding Categorical Variables

Machine learning algorithms require numerical input. We must convert categorical data!

In [ ]:
# ============================================================
# STEP 7: Identify Categorical Columns
# ============================================================

print("📋 Categorical Columns in Our Dataset:")
print("=" * 80)

categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    unique_vals = df_cleaned[col].unique()
    print(f"\n{col}:")
    print(f"  Unique values ({len(unique_vals)}): {unique_vals}")

In [ ]:
# ============================================================
# STEP 8: Label Encoding
# ============================================================
# Label Encoding: Converts categories to numbers (0, 1, 2, ...)
# Best for: Ordinal data (categories with natural order)
# Example: Low < Medium < High

print("🏷️ Label Encoding Example:")
print("=" * 80)

# Create LabelEncoder instance
le = LabelEncoder()

# Let's encode 'time_of_day' - it has a natural order
df_cleaned['time_of_day_encoded'] = le.fit_transform(df_cleaned['time_of_day'])

# Display the mapping
print("\nOriginal → Encoded:")
for i, label in enumerate(le.classes_):
    print(f"  {label} → {i}")

# Show sample of original vs encoded
print("\nSample comparison:")
df_cleaned[['time_of_day', 'time_of_day_encoded']].head(10)

In [ ]:
# ============================================================
# STEP 9: One-Hot Encoding
# ============================================================
# One-Hot Encoding: Creates binary columns for each category
# Best for: Nominal data (no natural order)
# Example: Colors (Red, Blue, Green - no inherent order)

print("🔥 One-Hot Encoding Example:")
print("=" * 80)

# Method 1: Using pandas get_dummies() - simpler approach
print("\n📌 Method 1: pandas get_dummies()")

# Encode 'city' column
city_encoded = pd.get_dummies(df_cleaned['city'], prefix='city')
print(f"\nOriginal 'city' column has {df_cleaned['city'].nunique()} unique values")
print(f"After One-Hot Encoding: {city_encoded.shape[1]} new columns created")

city_encoded.head()

In [ ]:
# ============================================================
# STEP 10: One-Hot Encoding with Scikit-learn
# ============================================================
# Method 2: Using sklearn OneHotEncoder - more control

print("\n📌 Method 2: sklearn OneHotEncoder")

# Initialize OneHotEncoder
# sparse_output=False returns a dense array (easier to work with)
# handle_unknown='ignore' handles new categories during prediction
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform
vehicle_encoded = ohe.fit_transform(df_cleaned[['vehicle_type']])

# Get feature names
feature_names = ohe.get_feature_names_out(['vehicle_type'])

# Create DataFrame for better visualization
vehicle_encoded_df = pd.DataFrame(vehicle_encoded, columns=feature_names)

print(f"\nOriginal 'vehicle_type' column:")
print(df_cleaned['vehicle_type'].head(5).tolist())
print(f"\nOne-Hot Encoded:")
vehicle_encoded_df.head(5)

In [ ]:
# ============================================================
# STEP 11: Complete One-Hot Encoding for All Categorical Columns
# ============================================================

# Columns to one-hot encode (nominal categories)
columns_to_encode = ['city', 'vehicle_type', 'payment_method']

# Apply one-hot encoding using pandas
df_encoded = pd.get_dummies(
    df_cleaned, 
    columns=columns_to_encode,
    prefix=columns_to_encode,
    drop_first=False  # Set to True to avoid multicollinearity
)

print("\n📊 Dataset After Encoding:")
print(f"Original shape: {df_cleaned.shape}")
print(f"After encoding: {df_encoded.shape}")
print(f"\nNew columns added: {df_encoded.shape[1] - df_cleaned.shape[1]}")

# Show new column names
print("\nAll columns after encoding:")
print(df_encoded.columns.tolist())

In [ ]:
# ============================================================
# 💡 YOUR TURN: Practice Exercise 2
# ============================================================

# TODO: One-Hot encode the 'payment_method' column using sklearn OneHotEncoder
# and display the first 5 rows

# Uncomment and complete:

# ohe_payment = OneHotEncoder(sparse_output=False)
# payment_encoded = ohe_payment.fit_transform(df_cleaned[['________']])
# print("Encoded shape:", payment_encoded.shape)
# print("Feature names:", ohe_payment.get_feature_names_out())

## Part 5: Feature Scaling

Different features have different scales. This can bias ML algorithms!

Two main techniques:
1. **Standardization (Z-score)**: Mean = 0, Std = 1
2. **Normalization (Min-Max)**: Range [0, 1]

In [ ]:
# ============================================================
# STEP 12: Visualize the Need for Scaling
# ============================================================

# Select numerical columns for scaling
numerical_cols = ['distance_km', 'duration_minutes', 'fare_pkr', 'driver_rating', 'surge_multiplier']

# Show current distribution
print("📊 Current Scale of Numerical Features:")
print("=" * 80)
df_cleaned[numerical_cols].describe()

In [ ]:
# Visualize the different scales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Box plots to show scale differences
df_cleaned[['distance_km', 'duration_minutes', 'fare_pkr']].boxplot(ax=axes[0])
axes[0].set_title('Before Scaling - Different Scales!', fontweight='bold')
axes[0].set_ylabel('Value')

# Histogram for fare (shows distribution)
axes[1].hist(df_cleaned['fare_pkr'], bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Fare Distribution (PKR)', fontweight='bold')
axes[1].set_xlabel('Fare')

# Histogram for distance
axes[2].hist(df_cleaned['distance_km'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[2].set_title('Distance Distribution (km)', fontweight='bold')
axes[2].set_xlabel('Distance')

plt.tight_layout()
plt.show()

print("\n⚠️ Notice: fare_pkr ranges from 100-25000, while distance_km is only 1-500")
print("This scale difference will bias distance-based algorithms like KNN!")

In [ ]:
# ============================================================
# STEP 13: Standardization (Z-score Scaling)
# ============================================================
# Formula: z = (x - mean) / std
# Result: mean = 0, std = 1
# Best for: Algorithms assuming normally distributed data

print("📐 Standardization (Z-score Scaling):")
print("=" * 80)

# Initialize StandardScaler
scaler_standard = StandardScaler()

# Fit and transform numerical columns
df_standardized = df_cleaned.copy()
df_standardized[numerical_cols] = scaler_standard.fit_transform(df_cleaned[numerical_cols])

print("\nAfter Standardization:")
print(df_standardized[numerical_cols].describe().round(3))

print("\n✅ Notice: mean ≈ 0, std ≈ 1 for all columns!")

In [ ]:
# ============================================================
# STEP 14: Normalization (Min-Max Scaling)
# ============================================================
# Formula: x_norm = (x - min) / (max - min)
# Result: All values in range [0, 1]
# Best for: Algorithms requiring bounded input (Neural Networks)

print("📏 Normalization (Min-Max Scaling):")
print("=" * 80)

# Initialize MinMaxScaler
scaler_minmax = MinMaxScaler()

# Fit and transform
df_normalized = df_cleaned.copy()
df_normalized[numerical_cols] = scaler_minmax.fit_transform(df_cleaned[numerical_cols])

print("\nAfter Normalization:")
print(df_normalized[numerical_cols].describe().round(3))

print("\n✅ Notice: min = 0, max = 1 for all columns!")

In [ ]:
# ============================================================
# STEP 15: Visualize Scaling Results
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original data
df_cleaned[['distance_km', 'duration_minutes', 'fare_pkr']].boxplot(ax=axes[0])
axes[0].set_title('Original Data', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Value')

# Standardized data
df_standardized[['distance_km', 'duration_minutes', 'fare_pkr']].boxplot(ax=axes[1])
axes[1].set_title('After Standardization', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Z-score')

# Normalized data
df_normalized[['distance_km', 'duration_minutes', 'fare_pkr']].boxplot(ax=axes[2])
axes[2].set_title('After Normalization', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Normalized Value [0,1]')

plt.tight_layout()
plt.show()

print("✅ Now all features are on comparable scales!")

## Part 6: Outlier Detection and Handling

Outliers can significantly impact model performance. Let's identify and handle them.

In [ ]:
# ============================================================
# STEP 16: Detect Outliers Using IQR Method
# ============================================================
# IQR = Q3 - Q1
# Outliers: values < Q1 - 1.5*IQR or > Q3 + 1.5*IQR

print("🔍 Outlier Detection using IQR Method:")
print("=" * 80)

def detect_outliers_iqr(data, column):
    """
    Detect outliers using IQR method.
    Returns: lower_bound, upper_bound, outlier_indices
    """
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    
    return lower_bound, upper_bound, outliers.index

# Check for outliers in key columns
columns_to_check = ['distance_km', 'fare_pkr', 'duration_minutes']

print("\nOutlier Analysis:")
print("-" * 60)

for col in columns_to_check:
    lower, upper, outlier_idx = detect_outliers_iqr(df_cleaned, col)
    print(f"\n{col}:")
    print(f"  Valid Range: [{lower:.2f}, {upper:.2f}]")
    print(f"  Number of outliers: {len(outlier_idx)}")
    if len(outlier_idx) > 0:
        print(f"  Outlier values: {df_cleaned.loc[outlier_idx, col].values[:5]}...")  # Show first 5

In [ ]:
# ============================================================
# STEP 17: Visualize Outliers
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(columns_to_check):
    # Box plot showing outliers
    axes[i].boxplot(df_cleaned[col], vert=True)
    axes[i].set_title(f'{col}\nOutliers as circles above/below', fontweight='bold')
    axes[i].set_ylabel('Value')
    
    # Add scatter for outlier points
    lower, upper, outlier_idx = detect_outliers_iqr(df_cleaned, col)
    axes[i].axhline(y=upper, color='r', linestyle='--', alpha=0.7, label=f'Upper: {upper:.1f}')
    axes[i].axhline(y=lower, color='r', linestyle='--', alpha=0.7, label=f'Lower: {lower:.1f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

print("🔴 Red dashed lines show the IQR bounds. Points outside are outliers.")

In [ ]:
# ============================================================
# STEP 18: Handle Outliers - Multiple Strategies
# ============================================================

print("🛠️ Outlier Handling Strategies:")
print("=" * 80)

df_no_outliers = df_cleaned.copy()

# Strategy 1: Cap/Clip outliers (Winsorization)
# Replace outliers with boundary values
print("\n📌 Strategy: Capping/Clipping outliers at boundaries")

for col in columns_to_check:
    lower, upper, _ = detect_outliers_iqr(df_cleaned, col)
    
    original_outliers = ((df_no_outliers[col] < lower) | (df_no_outliers[col] > upper)).sum()
    
    # Clip values to boundaries
    df_no_outliers[col] = df_no_outliers[col].clip(lower=lower, upper=upper)
    
    remaining_outliers = ((df_no_outliers[col] < lower) | (df_no_outliers[col] > upper)).sum()
    
    print(f"  {col}: {original_outliers} outliers → {remaining_outliers} outliers")

print("\n✅ Outliers have been capped to boundary values!")

In [ ]:
# ============================================================
# STEP 19: Z-score Method for Outlier Detection
# ============================================================
# Outliers: |z-score| > 3 (more than 3 standard deviations from mean)

print("\n🔍 Outlier Detection using Z-score Method:")
print("=" * 80)

from scipy import stats

def detect_outliers_zscore(data, column, threshold=3):
    """
    Detect outliers using Z-score method.
    Threshold: typically 2.5 or 3
    """
    z_scores = np.abs(stats.zscore(data[column].dropna()))
    outlier_mask = z_scores > threshold
    return outlier_mask.sum(), z_scores[outlier_mask]

print("\nZ-score Outlier Analysis (|z| > 3):")
print("-" * 60)

for col in columns_to_check:
    count, z_vals = detect_outliers_zscore(df_cleaned, col)
    print(f"{col}: {count} outliers detected")

## Part 7: Building a Complete Preprocessing Pipeline

Now let's combine everything into a reusable scikit-learn pipeline!

In [ ]:
# ============================================================
# STEP 20: Create Complete Preprocessing Pipeline
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("🔧 Building Complete Preprocessing Pipeline:")
print("=" * 80)

# Define column groups
numerical_features = ['distance_km', 'duration_minutes', 'fare_pkr', 'driver_rating', 'surge_multiplier']
categorical_features = ['city', 'vehicle_type', 'payment_method']

# Create preprocessing pipelines for each type

# Numerical pipeline: Impute missing → Scale
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Handle missing values
    ('scaler', StandardScaler())                     # Standardize
])

# Categorical pipeline: Impute missing → One-Hot Encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Handle missing values
    ('onehot', OneHotEncoder(handle_unknown='ignore'))     # One-Hot Encode
])

# Combine pipelines using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop'  # Drop columns not specified
)

print("✅ Pipeline created successfully!")
print("\nPipeline Structure:")
print("  Numerical: Imputation → StandardScaler")
print("  Categorical: Imputation → OneHotEncoder")

In [ ]:
# ============================================================
# STEP 21: Apply the Pipeline
# ============================================================

# Prepare features and target
X = df[numerical_features + categorical_features]  # Original data with issues
y = df['ride_completed']

# Split data first (BEFORE fitting the preprocessor)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Fit on training data, transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nAfter preprocessing:")
print(f"  Training features shape: {X_train_processed.shape}")
print(f"  Test features shape: {X_test_processed.shape}")

In [ ]:
# ============================================================
# STEP 22: Get Feature Names After Transformation
# ============================================================

# Get feature names from the pipeline
feature_names = (numerical_features + 
                 list(preprocessor.named_transformers_['cat']
                      .named_steps['onehot']
                      .get_feature_names_out(categorical_features)))

print("📋 Final Feature Names After Preprocessing:")
print("=" * 80)
for i, name in enumerate(feature_names):
    print(f"  {i+1}. {name}")

In [ ]:
# ============================================================
# STEP 23: Create DataFrame with Processed Features
# ============================================================

# Convert to DataFrame for easier inspection
X_train_df = pd.DataFrame(
    X_train_processed, 
    columns=feature_names
)

print("📊 Processed Training Data (first 5 rows):")
print("=" * 80)
X_train_df.head()

In [ ]:
# Verify the preprocessing worked
print("\n📈 Verification - Numerical Features Statistics:")
print("=" * 80)
print(X_train_df[numerical_features].describe().round(3))

print("\n✅ Numerical features are now standardized (mean ≈ 0, std ≈ 1)")
print("✅ Categorical features are now one-hot encoded (0 or 1)")
print("✅ No missing values remain in the processed data")

## Part 8: Summary and Best Practices

### 🎯 Key Takeaways

1. **Always explore your data first** - Understand distributions, missing values, and outliers
2. **Handle missing values appropriately** - Mean/median for numerical, mode for categorical
3. **Encode categorical variables** - Label Encoding for ordinal, One-Hot for nominal
4. **Scale features** - Standardization or Normalization depending on the algorithm
5. **Detect and handle outliers** - IQR or Z-score methods
6. **Use pipelines** - Reproducible, prevents data leakage, easier deployment

### ⚠️ Common Pitfalls to Avoid

- **Data Leakage**: Never fit the scaler on test data!
- **Wrong Encoding**: Don't use Label Encoding for nominal categories
- **Forgetting to Scale**: Many algorithms (KNN, SVM, Neural Networks) require scaling
- **Dropping Too Much**: Be careful about removing outliers - they might be valuable!

In [ ]:
# ============================================================
# 🏆 FINAL CHALLENGE: Complete Exercise
# ============================================================

print("🏆 Final Challenge:")
print("=" * 80)
print("""
Create a preprocessing pipeline for a NEW dataset:

1. Load the dataset (uncomment the code below)
2. Explore and identify issues
3. Build a ColumnTransformer pipeline
4. Apply to train/test split
5. Verify the output

Good luck! 🍀
""")

# Uncomment to load a practice dataset
# from sklearn.datasets import fetch_openml
# titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
# df_titanic = titanic.frame
# print(df_titanic.head())
# print(df_titanic.info())

---

## 📚 Additional Resources

- [Scikit-learn Preprocessing Documentation](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Feature Engineering Guide](https://www.kaggle.com/learn/feature-engineering)

---

**Congratulations!** 🎉 You've completed the Data Preprocessing Lab!

Next Lab: Classification Models (Logistic Regression & KNN)